# Lab | Data Structuring and Combining Data

## Challenge 1: Combining & Cleaning Data

In this challenge, we will be working with the customer data from an insurance company, as we did in the two previous labs. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv

But this time, we got new data, which can be found in the following 2 CSV files located at the links below.

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv

Note that you'll need to clean and format the new data.

Observation:
- One option is to first combine the three datasets and then apply the cleaning function to the new combined dataset
- Another option would be to read the clean file you saved in the previous lab, and just clean the two new files and concatenate the three clean datasets

In [1]:
# Your code goes here

In [2]:
import pandas as pd 

In [3]:
file1 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv"
file2 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv"
file3 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)
df3 = pd.read_csv(file3)


In [4]:
print(df1.shape)
print(df2.shape)
print(df3.shape)


(4008, 11)
(996, 11)
(7070, 11)


In [5]:
# Fix Column names

def fix_columns(df):
    df = df.copy()
    df.columns = df.columns.str.lower().str.replace(' ', '_')
    return df

df1 = fix_columns(df1)
df2 = fix_columns(df2)
df3 = fix_columns(df3)


In [6]:
# Renaming ST to State

df1 = df1.rename(columns={'st': 'state'})
df2 = df2.rename(columns={'st': 'state'})

In [7]:
# Combining datasets
combined_raw = pd.concat([df1,df2,df3], ignore_index=True)

In [8]:
# Cleaning and removing duplicates 

def clean_customer_data(df):
    df = df.copy()
    
    
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype(str).str.strip()
    
    
    if 'gender' in df.columns:
        df['gender'] = df['gender'].str.lower().replace({
            'm': 'male', 'male': 'male',
            'f': 'female', 'female': 'female'
        })
    
   
    numeric_cols = [
        'customer_lifetime_value',
        'income',
        'monthly_premium_auto',
        'number_of_open_complaints',
        'total_claim_amount'
    ]
    
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    
    df = df.drop_duplicates()
    
    return df


In [9]:
combined_clean = clean_customer_data(combined_raw)

/var/folders/t1/g1ptmxln4nx5k_rz9978lf340000gn/T/ipykernel_19734/832403852.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include='object').columns:


In [10]:
combined_clean.shape
combined_clean.head()
combined_clean.dtypes


customer                         str
state                            str
gender                           str
education                        str
customer_lifetime_value      float64
income                       float64
monthly_premium_auto         float64
number_of_open_complaints    float64
policy_type                      str
vehicle_class                    str
total_claim_amount           float64
dtype: object

# Challenge 2: Structuring Data

In this challenge, we will continue to work with customer data from an insurance company, but we will use a dataset with more columns, called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by performing data cleaning, formatting, and structuring.

In [11]:
# Your code goes here

In [12]:
import pandas as pd

file = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv"

df = pd.read_csv(file)


In [13]:
# verify 
df.head()
df.columns
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 10910 entries, 0 to 10909
Data columns (total 27 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   unnamed:_0                     10910 non-null  int64  
 1   customer                       10910 non-null  str    
 2   state                          10910 non-null  str    
 3   customer_lifetime_value        10910 non-null  float64
 4   response                       10910 non-null  str    
 5   coverage                       10910 non-null  str    
 6   education                      10910 non-null  str    
 7   effective_to_date              10910 non-null  str    
 8   employmentstatus               10910 non-null  str    
 9   gender                         10910 non-null  str    
 10  income                         10910 non-null  int64  
 11  location_code                  10910 non-null  str    
 12  marital_status                 10910 non-null  str    
 1

In [17]:
# Create Pivot table 

channel_revenue = df.pivot_table(
    values="total_claim_amount",
    index="sales_channel",
    aggfunc="sum"
).round(2)


channel_revenue


,total_claim_amount
sales_channel,
Agent,1810226.82
Branch,1301204.00
Call Center,926600.82
Web,706600.04


In [18]:
## Agent generates the highest total revenue
## Branch also generates high revenues 
## Web generates the lowest 

In [19]:
pivot_clv = df.pivot_table(
    values='customer_lifetime_value',
    index='gender',
    columns='education',
    aggfunc='mean'
).round(2)

pivot_clv


education,Bachelor,College,Doctor,High School or Below,Master
gender,,,,,
F,7874.27,7748.82,7328.51,8675.22,8157.05
M,7703.60,8052.46,7415.33,8149.69,8168.83


In [20]:
## Customer with higher education have higher Customer Lifetime Value
## Gender is relatively small -> Female show higher Customer Lifetime value 

In [21]:
clv_pivot = df.pivot_table(
    values="customer_lifetime_value",
    index="gender",
    columns="education",
    aggfunc="mean"
).round(2)

clv_pivot



education,Bachelor,College,Doctor,High School or Below,Master
gender,,,,,
F,7874.27,7748.82,7328.51,8675.22,8157.05
M,7703.60,8052.46,7415.33,8149.69,8168.83


1. You work at the marketing department and you want to know which sales channel brought the most sales in terms of total revenue. Using pivot, create a summary table showing the total revenue for each sales channel (branch, call center, web, and mail).
Round the total revenue to 2 decimal points.  Analyze the resulting table to draw insights.

2. Create a pivot table that shows the average customer lifetime value per gender and education level. Analyze the resulting table to draw insights.

## Bonus

You work at the customer service department and you want to know which months had the highest number of complaints by policy type category. Create a summary table showing the number of complaints by policy type and month.
Show it in a long format table.

*In data analysis, a long format table is a way of structuring data in which each observation or measurement is stored in a separate row of the table. The key characteristic of a long format table is that each column represents a single variable, and each row represents a single observation of that variable.*

*More information about long and wide format tables here: https://www.statology.org/long-vs-wide-data/*

In [22]:
# Your code goes here

In [23]:
import pandas as pd

file = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv"
df = pd.read_csv(file)

df.columns = df.columns.str.lower().str.replace(' ', '_')


In [24]:
df['effective_to_date'] = pd.to_datetime(df['effective_to_date'])
df['month'] = df['effective_to_date'].dt.month_name()


In [25]:
df[['effective_to_date', 'month']].head()


,effective_to_date,month
0,2011-02-18,February
1,2011-01-18,January
2,2011-02-10,February
3,2011-01-11,January
4,2011-01-17,January


In [26]:
df['effective_to_date'] = pd.to_datetime(df['effective_to_date'])
df['month'] = df['effective_to_date'].dt.month_name()



In [31]:
pivot_complaints = df.pivot_table(
    values='number_of_open_complaints',
    index='policy_type',
    columns='month',
    aggfunc='sum',
    fill_value=0
).round(2)

pivot_complaints



month,February,January
policy_type,,
Corporate Auto,385.21,443.43
Personal Auto,1453.68,1727.61
Special Auto,95.23,87.07


In [32]:
complaints_long = pivot_complaints.reset_index().melt(
    id_vars='policy_type',
    var_name='month',
    value_name='total_complaints'
)


In [33]:
complaints_long


,policy_type,month,total_complaints
0,Corporate Auto,February,385.21
1,Personal Auto,February,1453.68
2,Special Auto,February,95.23
3,Corporate Auto,January,443.43
4,Personal Auto,January,1727.61
5,Special Auto,January,87.07
